# CRIMEX Actor Intelligence Dataset Pipeline

This notebook builds the CRIMEX Actor Intelligence Dataset from public OpenSanctions data.

The pipeline downloads the OpenSanctions crime dataset, extracts person-level actors, applies reusable CRIMEX actor intelligence scripts, and saves a clean feature-engineered dataset.

The final dataset includes:

- actor identity features
- crime and charge text signals
- crime ontology mapping
- country and cross-border context
- source intelligence
- temporal observation features
- risk and investigator priority scores
- explainability fields

All reusable logic is stored in Python modules under `src/`, while this notebook acts as a reproducible orchestration layer.

In [3]:
# [1.1] Notebook start message

print("CRIMEX Actor Intelligence Dataset Pipeline")
print("Notebook started successfully.")

CRIMEX Actor Intelligence Dataset Pipeline
Notebook started successfully.


## [1.2] Setup project paths

In this step, we define the main project folders.

The notebook expects this structure:

project/
- notebooks/
- src/
- data/raw/
- data/final/

The notebook is inside the `notebooks` folder, so the project root is one folder above.

In [4]:
# [1.2] Setup project paths

from pathlib import Path
import sys

# Get the project root folder
PROJECT_ROOT = Path("..").resolve()

# Add the project root to Python path
# This allows the notebook to import files from src/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Define data folders
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "opensanctions"
FINAL_DIR = PROJECT_ROOT / "data" / "final" / "opensanctions"

# Create folders if they do not already exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Print paths for confirmation
print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Final folder:", FINAL_DIR)

Project root: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX
Raw folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\raw\opensanctions
Final folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\opensanctions


## [1.3] Import required libraries

In this step, we import the libraries needed for downloading data, reading CSV files, and saving outputs.

We also import the CRIMEX actor pipeline functions from the `src/` folder.

In [6]:
# [1.3] Import required libraries

import json
import requests
import pandas as pd

# Import reusable CRIMEX actor pipeline functions
from src.actor_pipeline import run_actor_pipeline
from src.actor_extended import add_actor_extended_features

print("Libraries imported successfully.")

Libraries imported successfully.


## [1.4] Download OpenSanctions catalog

OpenSanctions provides a public catalog that lists all available datasets.

We download the catalog first because it gives us the latest URL for the `crime` dataset.

This avoids hardcoding old dataset links.

In [7]:
# [1.4] Download OpenSanctions catalog

# URL of the latest OpenSanctions catalog
CATALOG_URL = "https://data.opensanctions.org/datasets/latest/index.json"

# Send request to download the catalog
response = requests.get(CATALOG_URL, timeout=60)

# Stop the notebook if the request failed
response.raise_for_status()

# Convert response to JSON
catalog = response.json()

# Print basic information
print("Catalog downloaded successfully.")
print("Catalog keys:", list(catalog.keys()))
print("Number of datasets:", len(catalog["datasets"]))

Catalog downloaded successfully.
Catalog keys: ['datasets', 'run_time', 'statements_url', 'model', 'target_topics', 'enrich_topics', 'schemata', 'app']
Number of datasets: 348


## [1.5] Select the OpenSanctions crime dataset

OpenSanctions contains many datasets.

For CRIMEX Actor Intelligence, we use the `crime` dataset because it contains warrants and criminal entities.

We select the `targets.simple.csv` file because it is flat, easier to process, and suitable for reproducible dataset building.

In [8]:
# [1.5] Select OpenSanctions crime dataset

# Find the dataset named "crime" in the catalog
crime_dataset = next(
    dataset for dataset in catalog["datasets"]
    if dataset["name"] == "crime"
)

# Extract all available resource URLs for this dataset
crime_resources = {
    resource["name"]: resource["url"]
    for resource in crime_dataset["resources"]
}

# Select the simple CSV file
CRIME_CSV_URL = crime_resources["targets.simple.csv"]

# Print dataset information
print("Selected dataset:", crime_dataset["title"])
print("Entity count:", crime_dataset["entity_count"])
print("Target count:", crime_dataset["target_count"])
print("CSV URL:", CRIME_CSV_URL)

Selected dataset: Warrants and Criminal Entities
Entity count: 511083
Target count: 247463
CSV URL: https://data.opensanctions.org/datasets/20260501/crime/targets.simple.csv


## [1.6] Download the raw dataset

This step downloads the OpenSanctions crime dataset.

If the file already exists locally, the notebook will not download it again.

This makes the notebook faster and safer to rerun.

In [9]:
# [1.6] Download raw OpenSanctions crime dataset

# Define where the downloaded CSV will be saved
crime_csv_path = RAW_DIR / "crime_targets_simple.csv"

# Download only if the file does not already exist
if not crime_csv_path.exists():
    print("Downloading dataset...")

    # Stream the download in chunks to avoid memory issues
    with requests.get(CRIME_CSV_URL, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(crime_csv_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    print("Download completed.")
else:
    print("File already exists. Skipping download.")

# Print file information
print("File path:", crime_csv_path)
print("File size MB:", round(crime_csv_path.stat().st_size / (1024 * 1024), 2))

Download completed.
File path: C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\raw\opensanctions\crime_targets_simple.csv
File size MB: 62.56


## [1.7] Load the raw dataset

Now we load the raw CSV file into a pandas DataFrame.

All columns are loaded as text to avoid mixed-type warnings and preserve identifiers exactly as they appear in the source.

In [10]:
# [1.7] Load raw dataset

# Load CSV using string types for safety
df_raw = pd.read_csv(
    crime_csv_path,
    dtype=str,
    low_memory=False
)

# Print dataset shape and columns
print("Raw dataset shape:", df_raw.shape)
print("Columns:")
print(df_raw.columns.tolist())

# Display first rows
df_raw.head()

Raw dataset shape: (247463, 16)
Columns:
['id', 'schema', 'name', 'aliases', 'birth_date', 'countries', 'addresses', 'identifiers', 'sanctions', 'phones', 'emails', 'program_ids', 'dataset', 'first_seen', 'last_seen', 'last_change']


,id,schema,name,aliases,birth_date,countries,addresses,identifiers,sanctions,phones,emails,program_ids,dataset,first_seen,last_seen,last_change
0,NK-223CQDBzp8MRkdJMDiqXn3,LegalEntity,YATAI SMART INDUSTRIAL NEW CITY,"MYANMAR YATAI INTERNATIONAL HOLDING GROUP CO.,...",NaN,mm,HPA-AN CITY,GQBPAV1TFF41;PW2USM6LNCJ9;PW2XZT68KVW9;PW3LMJ5...,Reciprocal - Active - 2025-09-08,NaN,NaN,NaN,US SAM Procurement Exclusions,2026-03-30T15:36:11,2026-04-30T07:55:03,2026-04-29T07:55:02
1,NK-224TRezPqwzhQZ37exWxtX,Person,SANAVBARI NIKITENKO,NaN,1992-06-28,ru;tj,NaN,NaN,"""participation in the activity of a terrorist ...",NaN,NaN,INTERPOL-RN,INTERPOL Red Notices,2024-03-04T17:09:51,2026-05-01T00:27:25,2025-10-06T18:27:01
2,NK-224cDeEU5DQiFJ4W3hbagh,Person,James O. Wilson Jr.,NaN,NaN,us,"Fayetteville, GA 30214",NaN,NonProcurement - Active - 1988-12-22,NaN,NaN,NaN,US SAM Procurement Exclusions,2026-03-30T15:36:11,2026-04-30T07:55:03,2026-04-30T07:55:03
3,NK-22BiLDQqi5mhKfDeYuqs23,Person,ARNITA LEFF,NaN,NaN,us,"BEACHWOOD, OH 44122",1538404843,Reciprocal - Active - 2021-05-20;Reciprocal - ...,NaN,NaN,NaN,US SAM Procurement Exclusions,2024-05-09T07:55:01,2026-04-30T07:55:03,2026-03-24T16:20:01
4,NK-22HtK7WrxZ2sU3rmhz6PuZ,Person,Michael KUAJIAN,Michael KUAJIEN DUER MAYOK,NaN,NaN,NaN,NaN,Reciprocal - Active - 2019-12-10,NaN,NaN,NaN,US SAM Procurement Exclusions,2026-03-30T15:36:11,2026-04-30T07:55:03,2026-03-30T15:36:11


## [1.8] Run core actor intelligence pipeline

This step applies the reusable CRIMEX actor pipeline.

The pipeline keeps person records and creates core intelligence features such as:

- offender identity fields
- crime text extraction
- crime ontology mapping
- country and age features
- source risk indicators
- temporal observation features
- risk score and risk level
- investigator priority level

In [11]:
# [1.8] Run core actor intelligence pipeline

# Apply reusable actor pipeline from src/actor_pipeline.py
df_core = run_actor_pipeline(df_raw)

# Print output shape
print("Core actor dataset shape:", df_core.shape)

# Preview selected important fields
df_core[
    [
        "offender_id",
        "offender_name",
        "source_dataset_name",
        "crime_ontology_code_v3",
        "actor_risk_score_v3",
        "actor_risk_level_v3",
        "investigator_priority_level_refined"
    ]
].head()

Running CRIMEX Actor Intelligence pipeline...
Input shape: (247463, 16)
Actor pipeline completed.
Output shape: (170097, 95)
Core actor dataset shape: (170097, 95)


,offender_id,offender_name,source_dataset_name,crime_ontology_code_v3,actor_risk_score_v3,actor_risk_level_v3,investigator_priority_level_refined
0,NK-224TRezPqwzhQZ37exWxtX,SANAVBARI NIKITENKO,INTERPOL Red Notices,violent,14.0,high,high
1,NK-224cDeEU5DQiFJ4W3hbagh,James O. Wilson Jr.,US SAM Procurement Exclusions,unknown,2.0,low,low
2,NK-22BiLDQqi5mhKfDeYuqs23,ARNITA LEFF,US SAM Procurement Exclusions,unknown,3.5,medium,medium
3,NK-22HtK7WrxZ2sU3rmhz6PuZ,Michael KUAJIAN,US SAM Procurement Exclusions,unknown,1.5,low,low
4,NK-22SrWxmRnUfLKXFvtGuJVB,Kathleen J King,US SAM Procurement Exclusions,unknown,4.0,medium,medium


## [1.9] Run extended research features

This step adds optional research and investigator-focused features.

These include:

- rich crime text flags
- identity density
- geographic complexity
- temporal persistence
- source signal strength
- manual review flags
- investigator segment

These features are useful for deeper analysis, dashboards, and research experiments.

In [12]:
# [1.9] Run extended actor features

# Apply extended feature module from src/actor_extended.py
df_final = add_actor_extended_features(df_core)

# Print final dataset shape
print("Final actor dataset shape:", df_final.shape)

# Show newly added extended columns
extended_columns = [col for col in df_final.columns if col not in df_core.columns]

print("Number of extended features:", len(extended_columns))
print("Extended features:")
print(extended_columns)

# Preview extended features
df_final[
    [
        "offender_name",
        "actor_risk_level_v3",
        "investigator_priority_level_refined",
        "needs_manual_review_flag",
        "investigator_segment"
    ]
].head()

Final actor dataset shape: (170097, 112)
Number of extended features: 17
Extended features:
['has_rich_crime_text_flag', 'short_crime_text_flag', 'alias_identifier_ratio', 'identity_density_score', 'high_identity_density_flag', 'address_country_ratio', 'high_geo_complexity_flag', 'short_observed_actor_flag', 'very_long_observed_actor_flag', 'stale_record_flag', 'strong_source_signal_flag', 'weak_source_signal_flag', 'risk_priority_mismatch_flag', 'needs_manual_review_flag', 'core_identity_available_score', 'data_sparse_flag', 'investigator_segment']


,offender_name,actor_risk_level_v3,investigator_priority_level_refined,needs_manual_review_flag,investigator_segment
0,SANAVBARI NIKITENKO,high,high,0,high_risk_monitor
1,James O. Wilson Jr.,low,low,0,low_risk_archive
2,ARNITA LEFF,medium,medium,0,medium_risk_watch
3,Michael KUAJIAN,low,low,0,low_risk_archive
4,Kathleen J King,medium,medium,0,medium_risk_watch


## [1.10] Validate final dataset

Before saving the dataset, we check:

- final number of rows and columns
- risk level distribution
- investigator priority distribution
- missing offender IDs
- duplicate offender IDs

This helps confirm the output is clean and ready for publishing.

In [13]:
# [1.10] Validate final dataset

# Print final shape
print("Final shape:", df_final.shape)

# Check missing IDs
missing_ids = df_final["offender_id"].isna().sum()

# Check duplicate IDs
duplicate_ids = df_final["offender_id"].duplicated().sum()

# Print validation checks
print("Missing offender_id:", missing_ids)
print("Duplicate offender_id:", duplicate_ids)

# Risk distribution
print("\nRisk level distribution:")
print(df_final["actor_risk_level_v3"].value_counts())

# Priority distribution
print("\nInvestigator priority distribution:")
print(df_final["investigator_priority_level_refined"].value_counts())

# Investigator segment distribution
print("\nInvestigator segment distribution:")
print(df_final["investigator_segment"].value_counts())

Final shape: (170097, 112)
Missing offender_id: 0
Duplicate offender_id: 0

Risk level distribution:
actor_risk_level_v3
low       84270
medium    75981
high       9846
Name: count, dtype: int64

Investigator priority distribution:
investigator_priority_level_refined
low       84270
medium    75981
high       9846
Name: count, dtype: int64

Investigator segment distribution:
investigator_segment
low_risk_archive     82735
medium_risk_watch    65120
manual_review        12396
high_risk_monitor     9121
urgent_high_risk       725
Name: count, dtype: int64


## [1.11] Save final dataset

The final dataset is saved as a Parquet file.

Parquet is preferred because it preserves data types better than CSV and is efficient for large datasets.

In [14]:
# [1.11] Save final dataset

# Define output path
final_dataset_path = FINAL_DIR / "crimex_actor_intelligence_v2_extended.parquet"

# Save final dataset
df_final.to_parquet(
    final_dataset_path,
    index=False
)

# Print confirmation
print("Final dataset saved:")
print(final_dataset_path)

print("Saved shape:", df_final.shape)

Final dataset saved:
C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\opensanctions\crimex_actor_intelligence_v2_extended.parquet
Saved shape: (170097, 112)


## [1.12] Save feature dictionary

A feature dictionary documents the final dataset columns.

For each feature, we save:

- feature name
- data type
- non-null count
- null count
- number of unique values
- sample value

This helps users understand and reuse the dataset.

In [15]:
# [1.12] Save feature dictionary

# Create list to hold feature metadata
feature_dictionary = []

# Loop through all columns
for column in df_final.columns:
    non_null_count = int(df_final[column].notna().sum())
    
    # Get a sample non-null value if available
    sample_value = (
        df_final[column].dropna().iloc[0]
        if non_null_count > 0
        else None
    )

    # Add feature metadata
    feature_dictionary.append({
        "feature_name": column,
        "data_type": str(df_final[column].dtype),
        "non_null_count": non_null_count,
        "null_count": int(df_final[column].isna().sum()),
        "unique_values": int(df_final[column].nunique(dropna=True)),
        "sample_value": sample_value
    })

# Convert to DataFrame
df_feature_dictionary = pd.DataFrame(feature_dictionary)

# Save feature dictionary
feature_dictionary_path = FINAL_DIR / "crimex_actor_feature_dictionary.csv"

df_feature_dictionary.to_csv(
    feature_dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Feature dictionary saved:")
print(feature_dictionary_path)

df_feature_dictionary.head()

Feature dictionary saved:
C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\opensanctions\crimex_actor_feature_dictionary.csv


,feature_name,data_type,non_null_count,null_count,unique_values,sample_value
0,id,object,170097,0,170097,NK-224TRezPqwzhQZ37exWxtX
1,schema,object,170097,0,1,Person
2,name,object,170097,0,166600,SANAVBARI NIKITENKO
3,aliases,object,10299,159798,9313,Michael KUAJIEN DUER MAYOK
4,birth_date,object,72098,97999,18175,1992-06-28


## [1.13] Save dataset metadata

This step saves general metadata about the dataset.

The metadata file is useful for GitHub, Hugging Face, and future reproducibility.

In [16]:
# [1.13] Save dataset metadata

# Create metadata dictionary
metadata = {
    "dataset_name": "CRIMEX Actor Intelligence Dataset v2",
    "source": "OpenSanctions crime dataset",
    "source_url": CRIME_CSV_URL,
    "raw_rows": int(df_raw.shape[0]),
    "final_rows": int(df_final.shape[0]),
    "final_columns": int(df_final.shape[1]),
    "output_file": str(final_dataset_path),
    "feature_dictionary": str(feature_dictionary_path),
    "pipeline_modules": [
        "src/actor.py",
        "src/actor_pipeline.py",
        "src/actor_extended.py"
    ],
    "license_note": "Derived from OpenSanctions data. Users must comply with OpenSanctions data licensing and attribution requirements."
}

# Define metadata path
metadata_path = FINAL_DIR / "crimex_actor_dataset_metadata.json"

# Save metadata as JSON
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

print("Metadata saved:")
print(metadata_path)

Metadata saved:
C:\Users\ayman\Documents\IdiomX\github_idiomX\CrimeX\data\final\opensanctions\crimex_actor_dataset_metadata.json


## [1.14] Reproducibility command

The same core pipeline can also be executed from the command line using:

```bash
python scripts/run_actor_pipeline.py \
  --input data/raw/opensanctions/crime_targets_simple.csv \
  --output data/final/opensanctions/crimex_actor_intelligence_v2_core.parquet
  ```


### Code cell
```python
# [1.14] Print reproducibility summary

print("Reproducibility summary")
print("-----------------------")
print("Raw input:", crime_csv_path)
print("Final dataset:", final_dataset_path)
print("Feature dictionary:", feature_dictionary_path)
print("Metadata:", metadata_path)
print("Core pipeline module: src/actor_pipeline.py")
print("Extended feature module: src/actor_extended.py")
```
